# 3 · eval_matrix — every checkpoint × every test set
Loads all runs from `runs_registry.json`, evaluates each on the **test split** of PKU, Kaggle and DeepPCB, and builds:
- the headline mAP50 matrix (6-class, `pin_hole→missing_hole` merged)
- the **clean-5** matrix (missing_hole excluded — the rigorous version)
- per-class AP50 tables and a summary chart

Everything is saved under `results/`. `import mr_yolo11` is **required** here — MR checkpoints can't unpickle without it.

In [ ]:
# --- project root resolution (same pattern as run1/run2 notebooks) ---
import os, sys, json
from pathlib import Path

def resolve_project_root() -> Path:
    env = os.environ.get("PCB_PROJECT_ROOT")
    if env:
        return Path(env).resolve()
    cwd = Path.cwd().resolve()
    if cwd.name == "experiments":
        return cwd.parent
    if (cwd / "experiments").is_dir():
        return cwd
    return cwd

PROJECT_ROOT = resolve_project_root()
EXP_DIR = PROJECT_ROOT / "experiments"
sys.path.insert(0, str(EXP_DIR))   # mr_yolo11.py + make_modrand_dataset.py live here
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import mr_yolo11   # REQUIRED before loading MR checkpoints (also patches fuse())
from ultralytics import YOLO
import pandas as pd

YAML_DIR = EXP_DIR / "yamls"
REGISTRY = EXP_DIR / "runs_registry.json"
RESULTS  = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

IMGSZ = 640
TEST_SETS = {"PKU": "pku.yaml", "Kaggle": "kaggle.yaml", "DeepPCB": "deeppcb.yaml"}
CLEAN5_EXCLUDE = "missing_hole"   # the merged pin_hole/missing_hole class

runs = json.loads(REGISTRY.read_text())
print("runs:", list(runs))

In [ ]:
# the full evaluation loop: |runs| x 3 test sets
rows = []
for run_id, info in runs.items():
    model = YOLO(info["best"])
    for ts_name, ts_yaml in TEST_SETS.items():
        r = model.val(data=str(YAML_DIR / ts_yaml), split="test",
                      imgsz=IMGSZ, verbose=False, plots=False)
        names = [r.names[c] for c in r.box.ap_class_index]   # classes present in this set
        ap50 = {n: float(a) for n, a in zip(names, r.box.ap50)}
        clean5 = [v for k, v in ap50.items() if k != CLEAN5_EXCLUDE]
        rows.append({"run": run_id, "test": ts_name,
                     "mAP50": float(r.box.map50),
                     "mAP50-95": float(r.box.map),
                     "mAP50_clean5": sum(clean5) / len(clean5) if clean5 else 0.0,
                     **{f"AP50_{k}": v for k, v in ap50.items()}})
        print(f"{run_id:24s} -> {ts_name:8s} mAP50={r.box.map50:.3f}")

df = pd.DataFrame(rows)
df.to_csv(RESULTS / "eval_raw.csv", index=False)
df.head()

In [ ]:
# headline matrix (6-class merged)
mat = df.pivot(index="run", columns="test", values="mAP50")[["PKU", "Kaggle", "DeepPCB"]].round(3)
mat.to_csv(RESULTS / "matrix_mAP50.csv")
mat

In [ ]:
# clean-5 matrix (missing_hole/pin_hole excluded) — the rigorous table
mat5 = df.pivot(index="run", columns="test", values="mAP50_clean5")[["PKU", "Kaggle", "DeepPCB"]].round(3)
mat5.to_csv(RESULTS / "matrix_mAP50_clean5.csv")
mat5

In [ ]:
# per-class AP50 on DeepPCB: shows WHICH defects each component rescues
focus = df[df.test == "DeepPCB"].set_index("run")
percls = focus[[c for c in focus.columns if c.startswith("AP50_")]].round(3)
percls.to_csv(RESULTS / "perclass_deeppcb.csv")
percls

In [ ]:
# asymmetry + transfer-gap summary (thesis numbers)
def cell(run, test):
    s = df[(df.run == run) & (df.test == test)]["mAP50"]
    return float(s.iloc[0]) if len(s) else float("nan")

pairs = [("A_pku_stock", "F_pku_full"), ("H_dpcb_stock", "I_dpcb_full")]
for base, full in pairs:
    if base in set(df.run) and full in set(df.run):
        for t in TEST_SETS:
            print(f"{t:8s}  {base}: {cell(base, t):.3f}  ->  {full}: {cell(full, t):.3f}"
                  f"   delta={cell(full, t) - cell(base, t):+.3f}")
        print()

In [ ]:
import matplotlib.pyplot as plt
ax = mat.plot.bar(figsize=(11, 4))
ax.set_ylabel("mAP50")
ax.set_title("Cross-dataset evaluation matrix")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(RESULTS / "matrix.png", dpi=200)
plt.show()

In [ ]:
# paste-ready for the thesis
print(mat.to_markdown())
print()
print(mat5.to_markdown())